# Hate Speech Detection — Full Pipeline Demo

Demonstrates the complete 3-layer pipeline:

**Layer 1 (Retriever)** — SBERT encodes the query and retrieves semantically similar precedents from a FAISS index  
**Layer 2 (RAC Classifier)** — A fine-tuned RoBERTa classifier predicts *hate* / *not hate* from the augmented input  
**Layer 3 (Explainer)** — LLaMA 3.3 generates a grounded, human-readable explanation anchored to the retrieved evidence

## 1. Imports

In [1]:
import sys, gc, time
import numpy as np
import pandas as pd
import psutil
import torch
import faiss
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, TimeoutError as FuturesTimeout

sys.path.insert(0, str(Path("..").resolve()))  # adds src/ so retriever.py and llm_explainer.py are importable
from llm_explainer import (
    explain, Layer2Output, ExplainerOutput,
    strip_label, _extract_label,
    load_pipeline, retrieve_numpy, run_pipeline, display_result, wrap,
)

/Users/alexandre/anaconda3/envs/dl_env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Layer 3 — LLM Backend

Choose your LLM backend. Groq is recommended (free tier, fast). Uncomment Option B or C for Ollama or OpenAI.

In [ ]:
# --- Option A: Groq (free, recommended) — set GROQ_API_KEY in your environment ---
# export GROQ_API_KEY="your_key_here"   (shell)  OR  set it in a .env file
import os
from groq import Groq

llm_client = Groq(api_key='your_key_here')  # replace with your actual API key
LLM_MODEL  = "llama-3.3-70b-versatile"



print(f"LLM backend ready: {LLM_MODEL}")

LLM backend ready: llama-3.3-70b-versatile


## Pipeline Configuration

Choose the classifier model, index split and training dataset. `INDEX_SPLIT` and `DATASET` must match the saved weights.

In [4]:
# Classifier and index configuration
MODEL_FAMILY = "roberta"   # "bert" | "roberta"
INDEX_SPLIT  = "full"      # "example" | "knowledge" | "full"
DATASET      = "ISHate"    # "IHC" | "ISHate"
K            = 3           # max neighbors to retrieve
THRESHOLD    = 0.3         # min cosine similarity (sbert is well-separated, not collapsed to ~0.997)

# Define paths (resolved to absolute so AutoTokenizer.from_pretrained accepts them)
ROOT_DIR        = Path("../..")
INDEX_DIR       = ROOT_DIR / "corpus" / "index"
WEIGHTS_RAC_DIR = ROOT_DIR / "weights" / "weights_rac_best_hyperparameters"

## Input Texts

Add the texts you want to analyse. No ground-truth labels needed.

In [5]:
# Add or edit the texts to analyse
TEXTS = [
    {"id": 1, "text": "You will be drafted to defend zog"},   
    # Add more entries as needed.
]

print(f"Loaded {len(TEXTS)} custom input(s).")
for t in TEXTS:
    print(f"  [{t['id']}] {t['text'][:100]}")

Loaded 1 custom input(s).
  [1] You will be drafted to defend zog


## Load Pipeline Components

Load the **SBERT retriever** (Layer 1), the **FAISS index**, and the **RAC classifier** (Layer 2).

In [6]:
ret_model, ret_tokenizer, index, documents, clf_model, clf_tokenizer, device = load_pipeline(
    MODEL_FAMILY, INDEX_SPLIT, DATASET, INDEX_DIR, WEIGHTS_RAC_DIR
)

Device: cpu
Loading retriever: sentence-transformers/all-mpnet-base-v2 ...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7217.24it/s]


Loading index: ../../corpus/index/vdb_full.faiss ...
  Index size: 108,816 vectors
Loading RAC classifier: ../../weights/weights_rac_best_hyperparameters/roberta/sbert/full/ISHate ...


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 7940.24it/s]


Ready: ROBERTA | index=sbert/full | trained_on=ISHate


## Layer 2 → Layer 3: Run the Full Pipeline

For each input text:
1. **Layer 1** — encode with SBERT and retrieve the top-*k* nearest neighbors from the FAISS index  
2. **Layer 2** — feed the augmented input (query + retrieved passages) to the RAC classifier  
3. **Layer 3** — pass the classification result and retrieved evidence to LLaMA for a grounded explanation

Results are stored in `records` for inspection below.

In [7]:
print(f"RAM available: {psutil.virtual_memory().available / 1e9:.1f} GB")
print(f"Config : {MODEL_FAMILY.upper()} | index=sbert/{INDEX_SPLIT} | trained_on={DATASET}")
print(f"Running pipeline on {len(TEXTS)} input(s).\n")

LLM_TIMEOUT = 30  # seconds per LLM call before giving up

# Extract raw index vectors for numpy-based cosine search (avoids FAISS at query time)
print("Extracting index vectors for numpy search...", end=" ", flush=True)
_inner  = faiss.downcast_index(index.index)
_xb     = np.empty((index.ntotal, index.d), dtype="float32")
_inner.reconstruct_n(0, index.ntotal, _xb)
_id_map = faiss.vector_to_array(index.id_map).astype("int64")
print(f"done  shape={_xb.shape}")

# Run pipeline on each input
records = []
t_start = time.time()

for i, entry in enumerate(TEXTS):
    text       = str(entry["text"])
    example_id = entry["id"]

    t0 = time.time()
    print(f"[{i+1:02d}/{len(TEXTS)}] id={example_id}  ...", end=" ", flush=True)

    # Layer 1 + 2: retrieve neighbors and classify
    retrieved = retrieve_numpy(text, THRESHOLD, K, ret_model, ret_tokenizer, _xb, _id_map, documents)

    sep       = clf_tokenizer.sep_token or "[SEP]"
    augmented = f" {sep} ".join([text] + [t for t, _ in retrieved])
    inputs    = clf_tokenizer(augmented, return_tensors="pt", truncation=True, padding=True, max_length=256)
    inputs    = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        logits = clf_model(**inputs).logits[0]
    del inputs
    probs      = torch.nn.functional.softmax(logits, dim=-1)
    pred_idx   = torch.argmax(probs).item()
    label      = "hate" if pred_idx == 1 else "not hate"
    confidence = probs[pred_idx].item()
    del logits, probs

    l2 = Layer2Output(
        original_text=text, label=label, confidence=confidence,
        hate_category="unknown", retrieved=retrieved,
    )

    # Layer 3: LLM explanation (with timeout)
    try:
        with ThreadPoolExecutor(max_workers=1) as ex:
            fut         = ex.submit(explain, l2, llm_client, LLM_MODEL)
            explanation = fut.result(timeout=LLM_TIMEOUT)
    except FuturesTimeout:
        print(f"TIMEOUT({LLM_TIMEOUT}s) ", end="", flush=True)
        explanation = ExplainerOutput(
            summary="LLM call timed out.", evidence_used=[], target_groups=[],
            severity="unknown", recommended_action="human-review",
            moderator_note="Groq call exceeded timeout.", validation_passed=False,
        )
    except Exception as e:
        print(f"ERR({e}) ", end="", flush=True)
        explanation = ExplainerOutput(
            summary=f"LLM error: {e}", evidence_used=[], target_groups=[],
            severity="unknown", recommended_action="human-review",
            moderator_note="LLM call raised an exception.", validation_passed=False,
        )

    elapsed = time.time() - t0
    print(f"pred={label.upper():8s}  conf={confidence:.1%}  {elapsed:.1f}s")

    records.append({
        "id"                : example_id,
        "text"              : text,
        "predicted"         : label,
        "confidence"        : round(confidence, 4),
        "n_retrieved"       : len(retrieved),
        "top_sim"           : round(retrieved[0][1], 4) if retrieved else None,
        "retrieved_passages": [
            {"text": strip_label(t), "label": _extract_label(t), "score": round(s, 4)}
            for t, s in retrieved
        ],
        "summary"           : explanation.summary,
        "evidence_used"     : explanation.evidence_used,
        "severity"          : explanation.severity,
        "action"            : explanation.recommended_action,
        "target_groups"     : explanation.target_groups,
        "moderator_note"    : explanation.moderator_note,
        "validation_passed" : explanation.validation_passed,
    })

    gc.collect()

results_df = pd.DataFrame(records)
total = time.time() - t_start
print(f"\nDone. {len(records)} input(s) processed  |  total={total/60:.1f} min")

RAM available: 4.2 GB
Config : ROBERTA | index=sbert/full | trained_on=ISHate
Running pipeline on 1 input(s).

Extracting index vectors for numpy search... done  shape=(108816, 768)
[01/1] id=1  ... 

Encoding: 100%|██████████| 1/1 [00:00<00:00,  1.26it/s]


pred=HATE      conf=99.2%  2.5s

Done. 1 input(s) processed  |  total=0.0 min


## Results — Overview

Key fields for each input at a glance: prediction, confidence, retrieval stats, severity, and recommended action.

In [8]:
print(f"Config : {MODEL_FAMILY.upper()} | index={INDEX_SPLIT} | trained_on={DATASET}")
print(f"Inputs : {len(records)}")
print("=" * 50)
display(results_df[["id", "predicted", "confidence", "n_retrieved", "top_sim", "severity", "action", "validation_passed"]])

Config : ROBERTA | index=full | trained_on=ISHate
Inputs : 1


,id,predicted,confidence,n_retrieved,top_sim,severity,action,validation_passed
0,1,hate,0.9917,3,0.4454,high,auto-block,True


## Results — Prediction & Action Breakdown

In [9]:
print(f"\nResults for {len(records)} input(s):")
print(f"{'ID':<6} {'Predicted':<12} {'Confidence':<12} {'Action'}")
print("-" * 50)
for r in records:
    conf_str = f"{r['confidence']:.1%}"
    print(f"  {str(r['id']):<4}  {r['predicted'].upper():<12}  {conf_str:<12}  {r['action']}")

action_counts = {}
for r in records:
    action_counts[r["action"]] = action_counts.get(r["action"], 0) + 1
print(f"\nAction breakdown: {action_counts}")
hate_count = sum(1 for r in records if r["predicted"] == "hate")
print(f"Predicted hate: {hate_count}/{len(records)}")


Results for 1 input(s):
ID     Predicted    Confidence   Action
--------------------------------------------------
  1     HATE          99.2%         auto-block

Action breakdown: {'auto-block': 1}
Predicted hate: 1/1


## Results — Detailed Walkthrough

Full output for each input: the original text, Layer 2 classification with retrieved neighbors, and the Layer 3 LLM explanation.

In [10]:
for r in records:
    print("=" * 80)
    print(wrap(f"[{r['id']}]  TEXT : {r['text']}"))
    print("-" * 80)
    print(f"LAYER 2  : {r['predicted'].upper()}  ({r['confidence']:.1%} confidence)")
    print()
    print(f"RETRIEVED NEIGHBORS ({r['n_retrieved']}):")
    for i, p in enumerate(r["retrieved_passages"], 1):
        line = f"  [{i}] {p['score']:.4f}  [{p['label']}]  {p['text']}"
        print(wrap(line, indent=14))
    print()
    print("LAYER 3 EXPLANATION:")
    print(wrap(f"  Summary   : {r['summary']}"))
    print(f"  Severity  : {r['severity']}")
    print(f"  Action    : {r['action']}")
    targets = ", ".join(r["target_groups"]) if r["target_groups"] else "—"
    print(wrap(f"  Targets   : {targets}"))
    evidence = ", ".join(str(e) for e in r["evidence_used"]) if r["evidence_used"] else "—"
    print(wrap(f"  Evidence  : {evidence}"))
    if r["moderator_note"]:
        print(wrap(f"  Note      : {r['moderator_note']}"))
    valid_str = "✓ passed" if r["validation_passed"] else "✗ FAILED (forced human-review)"
    print(f"  Validation: {valid_str}")
    print("=" * 80)
    print()

[1]  TEXT : You will be drafted to defend zog
--------------------------------------------------------------------------------
LAYER 2  : HATE  (99.2% confidence)

RETRIEVED NEIGHBORS (3):
  [1] 0.4454  [hate]  zog: Zionist Occupation Government, an organization
              hypothesized by white supremacists to rule the world. Target
              category: Judaism.
  [2] 0.4274  [hate]  this is antifa . zog will take anyone to fight for greater
              israel
  [3] 0.4191  [hate]  Maximum resistance towards Zog !

LAYER 3 EXPLANATION:
  Summary   : The content was flagged as hate speech due to the reference to
              'zog', a term associated with white supremacist ideology and anti-
              Semitic conspiracy theories. This term is often used to express
              hostility towards Jewish people and the state of Israel.
  Severity  : high
  Action    : auto-block
  Targets   : Jewish people, Israel
  Evidence  : 1, 2, 3
  Note      : The high confidence score a

## Results — Inspect Flagged Inputs

Filter results to explore specific predictions.

In [11]:
# Adjust the filter to explore any subset of results

flagged = [r for r in records if r["predicted"] == "hate"]
print(f"Flagged as hate: {len(flagged)}/{len(records)}\n")
for r in flagged:
    print(f"[{r['id']}] pred={r['predicted'].upper():8s}  conf={r['confidence']:.1%}  action={r['action']}")
    print(f"{r['text'][:120]}")

Flagged as hate: 1/1

[1] pred=HATE      conf=99.2%  action=auto-block
You will be drafted to defend zog
